# 3. Figures

Recreate the four manuscript figures from `results/`. No model training occurs here.
All four PNG files are exported at 600 dpi. Run from the supplement folder.

In [ ]:
%matplotlib inline
from pathlib import Path
ROOT = Path.cwd()
FIGURES = ROOT / "figures"
FIGURES.mkdir(exist_ok=True)

## Figure 1. Experimental design

Draw the protocol diagram from Matplotlib shapes and text. Reproducing the manuscript layout requires Arial Narrow (regular and bold). The PNG is exported at 600 dpi.


In [ ]:
from IPython.display import display
import os
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import font_manager
from matplotlib.patches import Circle, FancyArrowPatch, FancyBboxPatch, PathPatch, Polygon, Rectangle
from matplotlib.path import Path as DrawingPath
from PIL import Image


def plot_test_design_protocol(output):
    if "WINDIR" in os.environ:
        for filename in ("ARIALN.TTF", "ARIALNB.TTF"):
            font_path = Path(os.environ["WINDIR"]) / "Fonts" / filename
            if font_path.is_file():
                font_manager.fontManager.addfont(font_path)
    family, font_scale = "Arial Narrow", 1.0
    try:
        font_manager.findfont(font_manager.FontProperties(family=family), fallback_to_default=False)
    except ValueError:
        family, font_scale = "DejaVu Sans", .72
    with plt.rc_context({"font.family": family, "svg.fonttype": "none", "pdf.fonttype": 42, "savefig.bbox": None}):
        W, H = 1860, 1520
        WIDTH_MM = 190
        HEIGHT_MM = WIDTH_MM * H / W
        SCALE = WIDTH_MM / 25.4 * 72 / W * font_scale
        fig = plt.figure(figsize=(WIDTH_MM / 25.4, HEIGHT_MM / 25.4), facecolor="white")
        ax = fig.add_axes((0, 0, 1, 1), xlim=(0, W), ylim=(H, 0))
        ax.set_axis_off()
        GRAY, BLUE, ORANGE = "#D5DBE3", "#AFC5D9", "#F9C99D"
        INK, BORDER, BODY = "#181A1E", "#8F959E", "#F7FBFD"
        texts, containers = [], []


        def text(x, y, value, size=28, bold=False, align="center", color=INK, bounds=None):
            obj = ax.text(x, y, value, ha=align, va="center", color=color,
                          fontsize=size * SCALE, fontweight="bold" if bold else "normal",
                          linespacing=1.13, zorder=6)
            texts.append((obj, bounds))
            return obj


        def box(x, y, w, h, fill="white", radius=9, edge=BORDER, dashed=False, shadow=False):
            if shadow:
                ax.add_patch(FancyBboxPatch((x + 1.6, y + 2.5), w, h,
                             boxstyle=f"round,pad=0,rounding_size={radius}",
                             facecolor="#6C7784", edgecolor="none", alpha=.09, zorder=1))
            node = FancyBboxPatch((x, y), w, h,
                                  boxstyle=f"round,pad=0,rounding_size={radius}",
                                  facecolor=fill, edgecolor=edge, linewidth=.65,
                                  linestyle=(0, (3, 2)) if dashed else "solid", zorder=2)
            ax.add_patch(node)
            return node


        def card(x, y, w, h, title, color):
            base = box(x, y, w, h, fill=BODY, radius=12, shadow=True)
            band = Rectangle((x, y), w, 66, facecolor=color, edgecolor="none", zorder=3)
            band.set_clip_path(base)
            ax.add_patch(band)
            ax.plot([x, x + w], [y + 66, y + 66], color=BORDER, linewidth=.6, zorder=4)
            text(x + w / 2, y + 33, title, size=29 if "\n" in title else 31, bold=True,
                 bounds=(x + 8, y + 1, w - 16, 64))
            containers.append((x, y, w, h))


        def arrow(x, y, length, direction="right"):
            points = np.array([(0, -10), (length - 20, -10), (length - 20, -23),
                               (length, 0), (length - 20, 23), (length - 20, 10), (0, 10)])
            if direction == "down":
                points = points[:, ::-1]
            ax.add_patch(Polygon(points + [x, y], closed=True,
                                 facecolor=GRAY, edgecolor=BORDER, linewidth=.6, zorder=2))


        def elbow(points):
            (x0, y0), (_, y1), (x1, _) = points
            direction = 1 if y1 > y0 else -1
            path = DrawingPath([(x0, y0), (x0, y1 - direction * 12),
                                (x0, y1), (x0 + 12, y1), (x1 - 18, y1)],
                               [DrawingPath.MOVETO, DrawingPath.LINETO, DrawingPath.CURVE3,
                                DrawingPath.CURVE3, DrawingPath.LINETO])
            for color, width in ((BORDER, 16), (GRAY, 13)):
                ax.add_patch(PathPatch(path, facecolor="none", edgecolor=color,
                             linewidth=width * SCALE, capstyle="butt", zorder=2))
            ax.add_patch(Polygon([(x1 - 19, y1 - 17), (x1, y1), (x1 - 19, y1 + 17)],
                                 facecolor=GRAY, edgecolor=BORDER, linewidth=.6, zorder=3))


        def selector_icon(code, x, y):
            color = "#5C7C99"
            if code == "MFW":
                for i, width in enumerate((19, 14, 9)):
                    ax.add_patch(Rectangle((x, y + i * 7), width, 3.5,
                                 facecolor=color, edgecolor="none", zorder=5))
            elif code == "DF":
                for dx in (0, 7, 14):
                    ax.add_patch(Rectangle((x + dx, y), 6, 18,
                                 facecolor="#DFE9F1", edgecolor=color, linewidth=.45, zorder=5))
                ax.plot([x + 2, x + 6, x + 16], [y + 12, y + 16, y + 5], color=color, linewidth=.7, zorder=6)
            elif code == "C20":
                ax.add_patch(Polygon([(x, y), (x + 20, y), (x + 12, y + 9),
                                      (x + 12, y + 18), (x + 8, y + 20), (x + 8, y + 9)],
                                     closed=True, facecolor="#DCE7F0", edgecolor=color, linewidth=.6, zorder=5))
            elif code == "EDR":
                ax.plot([x, x + 20], [y + 9, y + 9], color=color, linewidth=.6, zorder=5)
                ax.plot([x + 3, x + 17], [y + 2, y + 2], color=color, linewidth=1.0, zorder=5)
                ax.plot([x + 2, x + 6, x + 10, x + 15, x + 19],
                        [y + 17, y + 13, y + 20, y + 14, y + 17], color=color, linewidth=.65, zorder=5)
            elif code == "ZMAD":
                ax.plot([x + 10, x + 10], [y, y + 20], color=color, linewidth=.6, zorder=5)
                for yy, xx in ((3, 2), (10, 17), (17, 5)):
                    ax.plot([x + 10, x + xx], [y + yy, y + yy], color=color, linewidth=.65, zorder=5)
                    ax.add_patch(Circle((x + xx, y + yy), 1.8, facecolor=color, edgecolor="none", zorder=6))
            else:
                assert code == "GMDZ"
                for xx, yy in ((2, 3), (18, 3), (10, 18)):
                    ax.add_patch(Circle((x + xx, y + yy), 2.1, facecolor=color, edgecolor="none", zorder=6))
                ax.plot([x + 2, x + 18, x + 10, x + 2], [y + 3, y + 3, y + 18, y + 3],
                        color=color, linewidth=.6, zorder=5)


        def rank_lists(x, y, count):
            for col in range(count):
                for row, width in enumerate((45, 36, 28)):
                    ax.add_patch(Rectangle((x + col * 62, y + row * 13), width, 5,
                                 facecolor=("#86A6C3", "#BA9BCE", "#91B9A5")[col],
                                 edgecolor="none", zorder=4))


        def gaussian(x, y):
            xx = np.linspace(-2.8, 2.8, 120)
            yy = np.exp(-xx ** 2 / 2)
            ax.plot(x + xx * 20, y - yy * 58, color="#7D9BB7", linewidth=.8, zorder=4)
            ax.fill_between(x + xx * 20, y - yy * 58, y, color="#D5E1EC", zorder=3)
            ax.plot([x - 62, x + 62], [y, y], color="#40464E", linewidth=.6, zorder=4)
            ax.plot([x, x], [y, y - 61], color="#7D9BB7", linewidth=.6, zorder=4)


        def classifiers(x, y):
            for index, (code, label) in enumerate((("D", "Delta"), ("C", "Cosine"),
                                                   ("L", "Logistic"), ("S", "Linear SVM"))):
                yy = y + 133 + index * 55
                text(x + 17, yy, f"{code}  {label}", size=28, align="left")
                gx, gy = x + 170, yy - 21
                blue, orange = "#6E94B4", "#D5A170"
                if code == "D":
                    ax.plot([gx + 6, gx + 58], [gy + 36, gy + 5], color="#BCC8D3",
                            linewidth=.6, linestyle=(0, (2, 2)), zorder=4)
                    ax.plot([gx + 6, gx + 58, gx + 58], [gy + 36, gy + 36, gy + 5],
                            color=blue, linewidth=.85, zorder=5)
                    ax.add_patch(Circle((gx + 6, gy + 36), 3, facecolor=blue, edgecolor="none", zorder=6))
                    ax.plot([gx + 54, gx + 62], [gy + 5, gy + 5], color=orange, linewidth=.9, zorder=6)
                    ax.plot([gx + 58, gx + 58], [gy + 1, gy + 9], color=orange, linewidth=.9, zorder=6)
                elif code == "C":
                    origin = (gx + 5, gy + 39)
                    for endpoint, color in (((gx + 62, gy + 26), blue), ((gx + 37, gy + 2), orange)):
                        ax.add_patch(FancyArrowPatch(origin, endpoint, arrowstyle="-|>",
                                     mutation_scale=4, color=color, linewidth=.8, zorder=5))
                    angles = np.linspace(.23, .86, 30)
                    ax.plot(origin[0] + 21 * np.cos(angles), origin[1] - 21 * np.sin(angles),
                            color="#717A83", linewidth=.65, zorder=5)
                elif code == "L":
                    ax.plot([gx + 3, gx + 65], [gy + 40, gy + 40], color="#727B84", linewidth=.5, zorder=4)
                    ax.plot([gx + 3, gx + 3], [gy + 40, gy + 1], color="#727B84", linewidth=.5, zorder=4)
                    ax.plot([gx + 3, gx + 65], [gy + 21, gy + 21], color="#B9C3CC",
                            linewidth=.5, linestyle=(0, (2, 2)), zorder=4)
                    tt = np.linspace(-3, 3, 60)
                    ax.plot(gx + 5 + (tt + 3) * 9.5, gy + 39 - 36 / (1 + np.exp(-tt)),
                            color=orange, linewidth=.9, zorder=5)
                else:
                    for shift, linestyle in ((-10, (0, (2, 2))), (0, "solid"), (10, (0, (2, 2)))):
                        ax.plot([gx + 18 + shift, gx + 46 + shift], [gy + 41, gy + 1],
                                color="#788A9A", linewidth=.7 if shift == 0 else .45,
                                linestyle=linestyle, zorder=4)
                    ax.scatter([gx + 8, gx + 18, gx + 16], [gy + 19, gy + 9, gy + 28],
                               s=2.8, c=blue, marker="o", linewidths=0, zorder=5)
                    ax.scatter([gx + 49, gx + 59, gx + 48], [gy + 31, gy + 21, gy + 12],
                               s=2.8, c=orange, marker="s", linewidths=0, zorder=5)


        def selectors(x, y):
            for row, pair in enumerate((("MFW", "DF"), ("C20", "EDR"), ("ZMAD", "GMDZ"))):
                for col, label in enumerate(pair):
                    xx, yy = x + 12 + 144 * col, y + 142 + 48 * row
                    box(xx, yy, 132, 40, fill="#EAF0F5", edge="#BAC8D5", radius=4)
                    selector_icon(label, xx + 8, yy + 11)
                    text(xx + 36, yy + 20, label, size=30, align="left")


        def zscore(x, y):
            text(x + 120, y + 119, "Training-Based\nZ-Scores", size=31, bold=True)
            # Native vector text and a fraction bar remain editable after Canva import.
            text(x + 51, y + 226, "z =", size=39)
            text(x + 158, y + 204, "TF − μ", size=36)
            ax.add_patch(Rectangle((x + 100, y + 227.7), 117, 2.6,
                         facecolor=INK, edgecolor="none", zorder=6))
            text(x + 158, y + 257, "σ", size=36)
            gaussian(x + 120, y + 404)
            text(x + 120, y + 451, "Fit μ and σ\non training texts", size=29, bold=True)


        xs = [318, 600, 942, 1224, 1546]
        widths = [240, 300, 240, 280, 290]
        ys = [160, 870]
        CARD_H = 500
        text(318, 35, "Six selectors × four models, where available, in both tests", size=30, bold=True, align="left")
        text(318, 100, "Test 1. Training-set perturbation", size=36, bold=True, align="left")
        text(318, 810, "Test 2. Vocabulary-size perturbation", size=36, bold=True, align="left")

        card(18, 446, 230, 240, "Corpus Input", GRAY)
        text(133, 544, "Three Corpora", size=30, bold=True)
        text(133, 620, "IMDb62\nFreDraCor\nRusLit9", size=29)
        arrow(133, 686, 48, direction="down")
        card(18, 734, 230, 230, "Outer Split", GRAY)
        text(133, 831, "5-Fold\nGrouped CV", size=26, bold=True)
        for index in range(5):
            ax.add_patch(Rectangle((59 + 30 * index, 873), 27, 28,
                         facecolor=BLUE if index == 4 else "#E4E9EE", edgecolor=BORDER, linewidth=.5, zorder=4))
        text(133, 934, "U: train   T: test", size=28)
        for color, width in ((BORDER, 16), (GRAY, 13)):
            ax.plot([248, 283], [849, 849], color=color, linewidth=width * SCALE,
                    solid_capstyle="butt", zorder=2)
        elbow([(283, 849), (283, 410), (318, 410)])
        elbow([(283, 849), (283, 1120), (318, 1120)])
        ax.add_patch(Circle((283, 849), 7, facecolor=GRAY, edgecolor=BORDER, linewidth=.5, zorder=4))

        for row, yy in enumerate(ys):
            titles = ("Training Samples" if row == 0 else "Training Set", "Vocabulary\nSelection",
                      "Standardization", "Model Fitting", "Output")
            for i, title in enumerate(titles):
                card(xs[i], yy, widths[i], CARD_H, title, GRAY if i == 0 else BLUE if i < 3 else ORANGE)
            for i in range(4):
                arrow(xs[i] + widths[i], yy + 250, xs[i + 1] - xs[i] - widths[i])
            text(xs[1] + 150, yy + 105, "Six Selection Methods", size=30, bold=True)
            selectors(xs[1], yy)
            zscore(xs[2], yy)
            text(xs[3] + 140, yy + 110, "Four Classifiers", size=31, bold=True)
            classifiers(xs[3] + 8, yy + 57)
            text(xs[3] + 140, yy + 448, "Refit for every\nselected vocabulary", size=30)
            text(xs[4] + 145, yy + 113, "Quality and\nProfile Stability", size=31, bold=True)
            box(xs[4] + 14, yy + 179, 262, 48, dashed=True, radius=4)
            text(xs[4] + 145, yy + 203, "BA: mean(A, B)" if row == 0 else "BA: 500 words", size=30)
            box(xs[4] + 14, yy + 270, 262, 40, fill="#FCE4CB", edge="#FCE4CB", radius=5)
            text(xs[4] + 145, yy + 290, "Top-50 Profiles", size=31, bold=True)
            rank_lists(xs[4] + (82 if row == 0 else 52), yy + 345, 2 if row == 0 else 3)
            box(xs[4] + 14, yy + 409, 262, 76, dashed=True, radius=4)
            text(xs[4] + 145, yy + 447, "RBO(A, B)" if row == 0 else "Mean RBO:\n500 vs 400 / 600",
                 size=30, bold=True, bounds=(xs[4] + 16, yy + 411, 258, 72))

        text(438, 282, "Paired Samples\nA and B", size=31, bold=True)
        for label, yy in (("A", 354), ("B", 408)):
            text(343, yy + 10, label, size=30, bold=True)
            box(366, yy, 160, 22, fill="#AFC5D9", edge="#91A7BC", radius=3)
            if label == "B":
                ax.add_patch(Rectangle((510, yy), 16, 22, facecolor=ORANGE, edgecolor="none", zorder=4))
        text(438, 551, "Each ≈90% of U\n≈10% replaced\nwithin authors", size=30)
        text(438, 711, "Ten pairs per fold;\nauthor counts preserved.", size=29)
        text(750, 479, "500 Words per Fit", size=32, bold=True)
        for xx, label in ((683, "A"), (812, "B")):
            box(xx - 33, 530, 66, 88, fill="white", edge="#8FA5B9", radius=4)
            text(xx, 552, label, size=30, bold=True)
            rank_lists(xx - 23, 575, 1)
        text(750, 711, "Rank and select separately\non A and B, for each method.", size=29)
        text(1062, 711, "Fit μ and σ separately\non A and B.", size=29)
        text(1364, 711, "Refit each classifier\non A and B.", size=29)
        text(1691, 711, "Evaluate both fits\non the same texts T.", size=29)

        text(438, 994, "Full Outer\nTraining Set U", size=31, bold=True)
        for dx, dy in ((10, -10), (5, -5), (0, 0)):
            box(400 + dx, 1068 + dy, 76, 84, fill="white", edge="#8FA5B9", radius=3)
        text(438, 1110, "U", size=36, bold=True)
        text(438, 1261, "Same training texts\nfor all three sizes", size=30)
        text(438, 1421, "Use the full training\npart of each fold.", size=29)

        text(750, 1173, "One Ranking per Method", size=30, bold=True)
        table_x, table_y, first_w, cell_w, row_h = 609, 1204, 120, 54, 38
        xx = table_x
        for label, width in (("Ranks", first_w), ("400", cell_w), ("500", cell_w), ("600", cell_w)):
            ax.add_patch(Rectangle((xx, table_y), width, 34, facecolor="#E3EBF2",
                         edgecolor="#8295A6", linewidth=.65, zorder=4))
            text(xx + width / 2, table_y + 17, label, size=29)
            xx += width
        membership = []
        for row, label in enumerate(("1–400", "401–500", "501–600")):
            yy = table_y + 34 + row * row_h
            ax.add_patch(Rectangle((table_x, yy), first_w, row_h, facecolor="white",
                         edgecolor="#A5B3C0", linewidth=.5, zorder=4))
            text(table_x + first_w / 2, yy + row_h / 2, label, size=29)
            for col in range(3):
                included = col >= row
                membership.append(included)
                xx = table_x + first_w + col * cell_w
                ax.add_patch(Rectangle((xx, yy), cell_w, row_h,
                             facecolor=("#D5E2EC", "#B9CDDE", "#94B3CD")[row] if included else "white",
                             edgecolor="#A5B3C0", linewidth=.5, zorder=4))
                if included:
                    ax.plot([xx + 18, xx + 24, xx + 35], [yy + 19, yy + 25, yy + 12],
                            color="#476A86", linewidth=.7, zorder=5)
        text(750, 1421, "Nested 400/500/600-word sets\nfor each selection method.", size=29)
        text(1062, 1421, "Fit on U; reuse statistics\nfor the shared words.", size=29)
        text(1364, 1421, "Refit each classifier\nfor each vocabulary size.", size=29)
        text(1691, 1421, "Compare all three fits\non the same texts T.", size=29)
        text(930, 1490, "RBO_EXT@50, p = 0.95.  BA = balanced accuracy.  All selection and fitting use training data only.",
             size=29, color="#3D454D")

        fig.canvas.draw()
        renderer, inv = fig.canvas.get_renderer(), ax.transData.inverted()
        for obj, bounds in texts:
            bb = obj.get_window_extent(renderer).transformed(inv)
            assert bb.x0 >= 4 and bb.x1 <= W - 4, (obj.get_text(), bb.bounds)
            assert min(bb.y0, bb.y1) >= 4 and max(bb.y0, bb.y1) <= H - 4, obj.get_text()
            if bounds:
                x, y, w, h = bounds
                assert bb.x0 >= x and bb.x1 <= x + w, (obj.get_text(), bb.bounds, bounds)
                assert min(bb.y0, bb.y1) >= y and max(bb.y0, bb.y1) <= y + h, obj.get_text()
        labels = [obj.get_text() for obj, _ in texts]
        for name in ("MFW", "DF", "C20", "EDR", "ZMAD", "GMDZ"):
            assert labels.count(name) == 2, name
        assert membership == [True, True, True, False, True, True, False, False, True]
        assert len(containers) == 12
        assert not any("10,000" in label or "10000" in label for label in labels)
        output.mkdir(parents=True, exist_ok=True)
        target = output / "figure1_test_design_protocol.png"
        fig.savefig(target, dpi=600, facecolor="white")
        with Image.open(target) as image:
            assert image.size == (4488, 3667)
            assert all(abs(dpi - 600) < .1 for dpi in image.info["dpi"])
        display(fig)
        plt.close(fig)


plot_test_design_protocol(FIGURES)


## Figure 2. Accuracy and profile stability

Three corpora by two tests. Points show means without confidence intervals; RBO
scales differ between panels. The four unavailable IMDb62/C20 configurations are
omitted. Test 2 displays balanced accuracy at 500 lemmas.


In [ ]:
%matplotlib inline
import csv
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator, ScalarFormatter
from matplotlib.transforms import Bbox
import numpy as np
from IPython.display import display

def make_accuracy_stability_composite(csv_path):
    """Build the six-panel manuscript figure directly from saved metrics."""
    SOURCE = Path(csv_path)
    CORPORA = ("IMDb62", "FreDraCor", "RusLit9")
    METHODS = ("D", "C", "L", "S")
    SELECTORS = ("MFW", "EDR", "ZMAD", "GMDZ", "C20", "DF")
    COLORS = dict(zip(METHODS, ("#0072B2", "#D55E00", "#009E73", "#7B3294")))
    MARKERS = dict(zip(METHODS, ("o", "s", "^", "D")))
    MODEL_NAMES = ("Delta L1", "Cosine", "Logistic regression", "Linear SVM")
    WIDTH_MM, HEIGHT_MM = 180, 240
    KEYS = (("ba_test1", "rbo_test1"), ("ba_test2", "rbo_test2"))

    with SOURCE.open(encoding="utf-8-sig", newline="") as handle:
        table = list(csv.DictReader(handle))
    assert len(table) == 72
    assert len({(r["corpus"], r["selector"], r["method"]) for r in table}) == 72
    available = [r for r in table if r["status"] == "complete"]
    assert len(available) == 68
    assert {(r["corpus"], r["selector"]) for r in table if r["status"] != "complete"} == {("IMDb62", "C20")}
    for row in available:
        for key in ("ba_test1", "rbo_test1", "ba_test2", "rbo_test2"):
            row[key] = float(row[key])
            assert np.isfinite(row[key]) and 0 <= row[key] <= 1

    def window(values):
        lower, upper = min(values), max(values)
        padding = max(.008, .12 * (upper - lower))
        return max(0, lower - padding), min(1, upper + padding)

    def place_labels(fig, ax, records):
        """Place selector numbers without moving the measured points."""
        fig.canvas.draw()
        renderer = fig.canvas.get_renderer()
        bounds = ax.get_window_extent(renderer).padded(-2)
        pixels_per_pt = fig.dpi / 72
        points = np.array([ax.transData.transform((r["x"], r["y"])) for r in records])
        sizes = []
        for record in records:
            probe = ax.text(0, 0, record["label"], fontsize=8.2)
            extent = probe.get_window_extent(renderer)
            sizes.append((extent.width + 3 * pixels_per_pt, extent.height + 2 * pixels_per_pt))
            probe.remove()
        occupied, annotations = [], []
        offsets = [(dx, dy) for step in range(5, 65, 5) for dy in (step, -step)
                   for dx in [0] + [sign * n for n in range(6, 85, 6) for sign in (-1, 1)]]
        order = sorted(range(len(records)), key=lambda i: (-points[i, 1], points[i, 0], records[i]["label"]))
        for index in order:
            point = points[index]
            width, height = sizes[index]
            candidates = []
            for dx, dy in offsets:
                center = point + np.array([dx, dy]) * pixels_per_pt
                box = Bbox.from_bounds(center[0] - width / 2, center[1] - height / 2, width, height)
                if not (bounds.contains(box.x0, box.y0) and bounds.contains(box.x1, box.y1)):
                    continue
                if any(box.overlaps(other) for other in occupied):
                    continue
                if any(box.padded(2 * pixels_per_pt).contains(*p) for p in points):
                    continue
                segment = np.linspace(point, center, 12)[1:-2]
                crossing = sum(any(other.contains(*p) for p in segment) for other in occupied)
                cost = dx * dx + 1.15 * dy * dy + crossing * 600
                candidates.append((cost, center, box))
            assert candidates, (ax.get_label(), records[index])
            _, center, box = min(candidates, key=lambda item: item[0])
            occupied.append(box)
            record = records[index]
            annotation = ax.annotate(record["label"], (record["x"], record["y"]),
                xytext=ax.transData.inverted().transform(center), ha="center", va="center",
                fontsize=8.2, color="#202020", zorder=5,
                bbox={"boxstyle": "round,pad=0.06", "fc": "white", "ec": "none"},
                arrowprops={"arrowstyle": "-", "color": "#858A91", "lw": .45,
                            "shrinkA": 1.0, "shrinkB": 2.3})
            annotation.set_gid("label_" + record["id"])
            annotations.append(annotation)
        fig.canvas.draw()
        boxes = [a.get_bbox_patch().get_window_extent(fig.canvas.get_renderer()) for a in annotations]
        assert all(bounds.contains(b.x0, b.y0) and bounds.contains(b.x1, b.y1) for b in boxes)
        assert not any(a.overlaps(b) for i, a in enumerate(boxes) for b in boxes[:i])
        return annotations

    with plt.rc_context({"font.family": "DejaVu Sans", "font.size": 8.2,
                         "svg.fonttype": "none", "pdf.fonttype": 42,
                         "axes.linewidth": .65, "axes.edgecolor": "#3D4249"}):
        fig = plt.figure(figsize=(WIDTH_MM / 25.4, HEIGHT_MM / 25.4), dpi=180, facecolor="white")
        fig.text(16 / WIDTH_MM, 231 / HEIGHT_MM, "Attribution quality and profile stability",
                 fontsize=11, fontweight="bold", va="center")
        handles = [Line2D([], [], marker=MARKERS[m], linestyle="", markersize=6,
                          markerfacecolor=COLORS[m], markeredgecolor="#34383E", markeredgewidth=.5,
                          label=name) for m, name in zip(METHODS, MODEL_NAMES)]
        fig.legend(handles=handles, loc="center", bbox_to_anchor=(.54, 20 / HEIGHT_MM),
                   ncol=4, frameon=False, fontsize=9, columnspacing=1.4, handletextpad=.6)
        selector_handles = [Line2D([], [], linestyle="", color="none", label=f"{i}  {name}")
                            for i, name in enumerate(SELECTORS, 1)]
        fig.legend(handles=selector_handles, loc="center", bbox_to_anchor=(.54, 8.5 / HEIGHT_MM),
                   ncol=6, frameon=False, fontsize=8.8, handlelength=0, handletextpad=0,
                   columnspacing=1.7, title="Vocabulary selector (number beside each point)",
                   title_fontsize=8.5)
        for x, title, subtitle in ((16, "Test 1: training-corpus perturbation", "Balanced accuracy: mean A/B"),
                                   (106, "Test 2: vocabulary-size perturbation", "Balanced accuracy: 500 words")):
            fig.text(x / WIDTH_MM, 214 / HEIGHT_MM, title, fontsize=9.0, fontweight="bold", va="center")
            fig.text(x / WIDTH_MM, 208 / HEIGHT_MM, subtitle, fontsize=7.8, color="#4D535B", va="center")
        axes, exported, annotations = [], [], []
        for row_index, (corpus, bottom) in enumerate(zip(CORPORA, (156, 99, 42))):
            local = [r for r in available if r["corpus"] == corpus]
            info = local[0]
            accuracy = window([r[k] for r in local for k in ("ba_test1", "ba_test2")])
            fig.text(16 / WIDTH_MM, (bottom + 47.5) / HEIGHT_MM,
                     f"({chr(97 + row_index)})  {corpus}", fontsize=9.8, fontweight="bold", va="center")
            fig.text(176 / WIDTH_MM, (bottom + 47.5) / HEIGHT_MM,
                     f"{int(info['documents']):,} texts · {info['authors']} authors", fontsize=8.2,
                     color="#4D535B", ha="right", va="center")
            for test_index, left in enumerate((16, 106)):
                ax = fig.add_axes((left / WIDTH_MM, bottom / HEIGHT_MM, 70 / WIDTH_MM, 42 / HEIGHT_MM),
                                  label=f"{corpus}_test{test_index + 1}")
                axes.append(ax)
                ax.set(xlim=accuracy, ylim=window([r[KEYS[test_index][1]] for r in local]))
                ax.spines[["top", "right"]].set_visible(False)
                for axis in (ax.xaxis, ax.yaxis):
                    axis.set_major_locator(MaxNLocator(nbins=6, min_n_ticks=4))
                    axis.set_major_formatter(ScalarFormatter(useOffset=False))
                ax.tick_params(axis="both", labelsize=7.8, length=2.3, width=.55, pad=2)
                ax.grid(color="#DCDCDC", linewidth=.55, zorder=0)
                records = []
                for record in local:
                    xkey, ykey = KEYS[test_index]
                    x, y = record[xkey], record[ykey]
                    identity = f"{corpus}_test{test_index + 1}_{record['method']}_{record['selector']}"
                    point = ax.scatter(x, y, c=COLORS[record["method"]], marker=MARKERS[record["method"]],
                                       s=24, edgecolors="#34383E", linewidths=.55, zorder=3)
                    point.set_gid(identity)
                    np.testing.assert_allclose(point.get_offsets(), [[x, y]], rtol=0, atol=0)
                    records.append({"x": x, "y": y, "label": str(SELECTORS.index(record["selector"]) + 1),
                                    "id": identity})
                    exported.append((identity, ax, x, y))
                annotations.extend(place_labels(fig, ax, records))
        fig.text(3 / WIDTH_MM, 124 / HEIGHT_MM, "Profile stability (RBO)", rotation=90,
                 fontsize=9.3, ha="center", va="center")
        fig.text(.54, 31 / HEIGHT_MM, "Balanced accuracy", fontsize=9.3, ha="center", va="center")
        assert len(exported) == len(annotations) == 136
        fig.canvas.draw()
        renderer = fig.canvas.get_renderer()
        assert all(fig.bbox.contains(*corner) for t in fig.texts
                   for corner in (t.get_window_extent(renderer).p0, t.get_window_extent(renderer).p1))
        for i in range(0, 6, 2):
            np.testing.assert_array_equal(axes[i].get_xlim(), axes[i + 1].get_xlim())
        return fig


In [ ]:
figure = make_accuracy_stability_composite(ROOT / "results/metrics.csv")
figure.savefig(FIGURES / "figure2_accuracy_stability_all_corpora.png", dpi=600,
               bbox_inches=None, facecolor="white")
display(figure)
plt.close(figure)


## Figure 3. MFW contribution-rank displacement

Draw the two tests across the three corpora from `results/MFW_rank_displacements.csv`. The notebook exports a 600 dpi PNG and an editable SVG.


In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import NullLocator, StrMethodFormatter
import numpy as np
import pandas as pd

from IPython.display import display


def plot_mfw_rank_displacement(root):
    SOURCE = root / 'results/MFW_rank_displacements.csv'
    OUTPUT = root / 'figures'
    AUTHORS = {'IMDb62': 62, 'FreDraCor': 87, 'RusLit9': 9}
    TESTS = ('test1', 'test2')
    HEADERS = ('Test 1: training perturbation\nA vs B, 500 words',
               'Test 2: vocabulary size\nMean of 500/400 and 500/600')
    MODELS = ('D', 'C', 'L', 'S')
    COLORS = dict(zip(MODELS, ('#0072B2', '#D55E00', '#009E73', '#7B3294')))
    STYLES = dict(zip(MODELS, ('-', '--', '-.', ':')))
    MODEL_NAMES = {'D': 'D: L1 Delta', 'C': 'C: Cosine Delta',
                   'L': 'L: Logistic regression', 'S': 'S: Linear SVM'}
    TICKS = (1, 2, 5, 10, 20, 50, 100, 200, 500)
    data = pd.read_csv(SOURCE)
    limit = np.ceil(data.mean_abs_shift.max() * 1.08 / 25) * 25
    OUTPUT.mkdir(parents=True, exist_ok=True)
    style = {'font.family': 'DejaVu Sans', 'font.size': 8,
             'xtick.labelsize': 7, 'ytick.labelsize': 7,
             'axes.edgecolor': '#555555', 'axes.linewidth': .55,
             'svg.fonttype': 'none', 'savefig.bbox': None}
    with plt.rc_context(style):
        fig, axes = plt.subplots(3, 2, figsize=(165 / 25.4, 180 / 25.4),
                                 sharex=True, sharey=True, layout='constrained')
        fig.suptitle('MFW contribution-rank stability', fontsize=10, fontweight='bold')
        for row, (corpus, authors) in enumerate(AUTHORS.items()):
            for col, test in enumerate(TESTS):
                ax = axes[row, col]
                for model in MODELS:
                    frame = data.query('corpus == @corpus and test == @test and method == @model')
                    frame = frame.sort_values('position')
                    np.testing.assert_array_equal(frame.position, np.arange(1, 501))
                    assert np.isfinite(frame.mean_abs_shift).all()
                    ax.plot(frame.position, frame.mean_abs_shift, color=COLORS[model],
                            linestyle=STYLES[model], linewidth=.75, alpha=.95)
                ax.set(xscale='log', xlim=(1, 500), ylim=(0, limit),
                       yticks=(0, 50, 100, 150))
                ax.set_xticks(TICKS)
                ax.xaxis.set_major_formatter(StrMethodFormatter('{x:g}'))
                ax.xaxis.set_minor_locator(NullLocator())
                ax.grid(color='#DDDDDD', linewidth=.4)
                ax.set_axisbelow(True)
                ax.spines[['top', 'right']].set_visible(False)
                letter = chr(ord('A') + row * 2 + col)
                ax.text(.02, .97, f'{letter}  {corpus} ({authors} authors)',
                        transform=ax.transAxes, ha='left', va='top', fontsize=8,
                        fontweight='bold')
                if row == 0:
                    ax.set_title(HEADERS[col], fontsize=8.3, pad=7)
        fig.supxlabel('Word position in the reference ranking (log scale)', fontsize=8.5)
        fig.supylabel('Mean absolute rank displacement (positions)', fontsize=8.5)
        handles = [Line2D([], [], color=COLORS[m], linestyle=STYLES[m],
                          linewidth=1.3, label=MODEL_NAMES[m]) for m in MODELS]
        legend = fig.legend(handles=handles, ncol=4, loc='outside lower center', frameon=False,
                            fontsize=7.2, handlelength=1.8, columnspacing=1.1, handletextpad=.5)
        # Keep the shared x label and legend on separate lines at the approved figure size.
        height = fig.get_figheight() * 72
        fig.get_layout_engine().set(rect=(0, 55 / height, 1, 1 - 55 / height))
        fig._supxlabel.set_y(31 / height)
        fig._supxlabel.set_in_layout(False)
        legend.set_loc('lower center')
        legend.set_bbox_to_anchor((.5, 3 / height))
        legend.set_in_layout(False)
        fig.canvas.draw()
        renderer = fig.canvas.get_renderer()
        assert not fig._supxlabel.get_window_extent(renderer).overlaps(
            legend.get_window_extent(renderer))
        fig.savefig(OUTPUT / 'figure_MFW_rank_displacement.png', dpi=600, facecolor='white')
        display(fig)
        plt.close(fig)


plot_mfw_rank_displacement(ROOT)


## Figure 4. MFW contribution shares by rank

Draw the sorted MFW 500-word contribution shares from `results/MFW_contribution_shares.csv`. The notebook exports a 600 dpi PNG.


In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import NullLocator, PercentFormatter, StrMethodFormatter
import numpy as np
import pandas as pd

from IPython.display import display


def plot_mfw_contribution_share(root):
    CORPORA = ('IMDb62', 'FreDraCor', 'RusLit9')
    MODELS = ('D', 'C', 'L', 'S')
    LABELS = ('D · L1 Delta', 'C · Cosine Delta', 'L · Logistic regression', 'S · Linear SVM')
    COLORS = ('#0072B2', '#D55E00', '#008560', '#7B3294')
    STYLES = ('-', '--', '-.', ':')
    TICKS = (1, 2, 5, 10, 20, 50, 100, 500)
    data = pd.read_csv(root / 'results/MFW_contribution_shares.csv')
    assert len(data) == 3 * 4 * 500
    with mpl.rc_context({'font.family': 'DejaVu Sans', 'font.size': 7.2,
                         'axes.linewidth': .6, 'savefig.bbox': None}):
        fig, axes = plt.subplots(1, 3, figsize=(165/25.4, 65/25.4), sharey=True)
        fig.subplots_adjust(left=.104, right=.979, bottom=.25, top=.79, wspace=.16)
        fig.suptitle('MFW contribution shares by rank', fontsize=10.5, y=.975)
        for ax, corpus in zip(axes, CORPORA):
            subset = data.loc[data.corpus.eq(corpus)]
            for model, label, color, line in zip(MODELS, LABELS, COLORS, STYLES):
                curve = subset.loc[subset.model.eq(model)].sort_values('depth')
                np.testing.assert_array_equal(curve.depth, np.arange(1, 501))
                assert curve.share_percent.between(0, 5).all()
                np.testing.assert_allclose(curve.share_percent.sum(), 100, atol=1e-10)
                ax.plot(curve.depth, curve.share_percent, color=color, linestyle=line,
                        linewidth=1.15, label=label)
            ax.set_title(corpus, fontsize=8.5, fontweight='bold')
            ax.set(xscale='log', xlim=(.93, 550), xticks=TICKS,
                   ylim=(0, 5), yticks=np.arange(6))
            ax.xaxis.set_minor_locator(NullLocator())
            ax.xaxis.set_major_formatter(StrMethodFormatter('{x:g}'))
            ax.yaxis.set_major_formatter(PercentFormatter(xmax=100, decimals=0))
            ax.tick_params(length=2.5, pad=2)
            ax.grid(color='#DDDDDD', linewidth=.45)
            ax.set_axisbelow(True)
            ax.spines[['top', 'right']].set_visible(False)
        axes[0].set_ylabel('Share of total absolute contribution', fontsize=7.5)
        fig.supxlabel('Word rank by contribution magnitude (log scale)', y=.133, fontsize=8)
        handles, labels = axes[0].get_legend_handles_labels()
        fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(.5, .012),
                   frameon=False, ncols=4, fontsize=7.3, columnspacing=1.4)
        target = root / 'figures/figure_MFW_contribution_share.png'
        target.parent.mkdir(exist_ok=True)
        fig.savefig(target, dpi=600, facecolor='white')
        display(fig)
        plt.close(fig)


plot_mfw_contribution_share(ROOT)
